# K-Nearest Neighbors on Diabetes Dataset
**Goal:** Predict whether a patient has diabetes (1) or not (0)  
**Metrics:** Confusion Matrix | Accuracy | Error Rate | Precision | Recall

In [ ]:
# Import necessary libraries for data manipulation, numerical operations, and visualization
import pandas as pd           # Data loading and manipulation
import numpy as np            # Numerical operations and array handling
import matplotlib.pyplot as plt  # Basic plotting
import seaborn as sns         # Advanced statistical visualizations

## STEP 1 – Load Dataset

In [ ]:
# Load the diabetes dataset from CSV file
# Dataset contains medical predictor variables and one target variable (Outcome)
df = pd.read_csv('diabetes.csv')

# Print shape to understand dataset dimensions (rows, columns)
print('Shape:', df.shape)

# Display first 5 rows to understand data structure
df.head()

In [ ]:
# Display detailed information about dataset
# Shows: column names, non-null counts, data types, memory usage
df.info()

In [ ]:
# Generate statistical summary for all numerical columns
# Provides: count, mean, std, min, 25%, 50%, 75%, max values
df.describe()

## STEP 2 – Explore Class Distribution

In [ ]:
# Count the number of instances for each class (0 = No Diabetes, 1 = Diabetes)
# This helps identify class imbalance in the dataset
print(df['Outcome'].value_counts())

# Visualize the class distribution using a bar plot
# Different colors for each class for better visualization
df['Outcome'].value_counts().plot(kind='bar', color=['steelblue','tomato'], edgecolor='black')
plt.xticks([0,1], ['No Diabetes (0)', 'Diabetes (1)'], rotation=0)
plt.title('Class Distribution')
plt.ylabel('Count')
plt.show()

## STEP 3 – Handle Missing / Zero Values
Columns like Glucose, BloodPressure, BMI cannot be 0 — replace zeros with column mean.

In [ ]:
# Medical features cannot realistically be zero
# Replace zero values with the mean of respective columns
# This is a simple imputation strategy for missing/invalid data
zero_cols = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
for col in zero_cols:
    df[col] = df[col].replace(0, df[col].mean())

# Verify no missing values remain after cleaning
print('Missing values after cleaning:')
print(df.isnull().sum())

## STEP 4 – Correlation Heatmap

In [ ]:
# Create correlation matrix heatmap to visualize relationships between features
# Helps identify which features are strongly correlated with the target
plt.figure(figsize=(10,7))
sns.heatmap(df.corr(), annot=True, fmt='.2f', cmap='coolwarm', linewidths=0.5)
plt.title('Feature Correlation Heatmap')
plt.show()

## STEP 5 – Feature / Target Split & Train-Test Split

In [ ]:
# Import train_test_split for dividing data into training and testing sets
from sklearn.model_selection import train_test_split

# Separate features (X) and target variable (y)
X = df.drop(columns=['Outcome'])  # All columns except Outcome
y = df['Outcome']                 # Target variable only

# Split into training (80%) and testing (20%) sets
# random_state ensures reproducible splits
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print('Train size:', X_train.shape[0])
print('Test  size:', X_test.shape[0])

## STEP 6 – Feature Scaling
KNN is distance-based — scaling is essential.

In [ ]:
# Import StandardScaler for feature normalization
from sklearn.preprocessing import StandardScaler

# Initialize scaler and fit on training data
scaler  = StandardScaler()
X_train = scaler.fit_transform(X_train)  # Fit and transform training data
X_test  = scaler.transform(X_test)       # Only transform test data (no fitting)

# Scaling ensures all features have mean=0 and std=1
# Critical for KNN since it uses distance calculations

## STEP 7 – Find Optimal K using Error Rate Plot

In [ ]:
# Import KNN classifier
from sklearn.neighbors import KNeighborsClassifier

# Test different K values (1 to 25) to find optimal number of neighbors
error_rates = []
for k in range(1, 26):
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train, y_train)           # Train model with current K
    preds = knn.predict(X_test)         # Make predictions on test set
    error_rates.append(np.mean(preds != y_test))  # Calculate error rate

# Plot error rate vs K value to visualize optimal K
plt.figure(figsize=(10, 5))
plt.plot(range(1, 26), error_rates, marker='o', color='steelblue')
plt.title('Error Rate vs K Value')
plt.xlabel('K')
plt.ylabel('Error Rate')
plt.grid()
plt.show()

# Find K with minimum error rate
best_k = error_rates.index(min(error_rates)) + 1
print(f'Best K = {best_k}  (Error Rate = {min(error_rates):.4f})')

## STEP 8 – Train KNN with Best K

In [ ]:
# Train final KNN model with optimal K value
knn = KNeighborsClassifier(n_neighbors=best_k)
knn.fit(X_train, y_train)       # Train on scaled training data
y_pred = knn.predict(X_test)    # Generate predictions on test set

## STEP 9 – Evaluation Metrics

In [ ]:
# Import evaluation metrics from sklearn
from sklearn.metrics import (
    confusion_matrix, ConfusionMatrixDisplay,
    accuracy_score, precision_score, recall_score, classification_report
)

# Calculate confusion matrix and evaluation metrics
cm        = confusion_matrix(y_test, y_pred)      # 2x2 matrix of predictions
accuracy  = accuracy_score(y_test, y_pred)        # Overall correctness
error     = 1 - accuracy                           # Overall error rate
precision = precision_score(y_test, y_pred)       # Positive predictive value
recall    = recall_score(y_test, y_pred)          # Sensitivity/True positive rate

# Display all metrics
print(f'Accuracy   : {accuracy:.4f}')
print(f'Error Rate : {error:.4f}')
print(f'Precision  : {precision:.4f}')
print(f'Recall     : {recall:.4f}')
print()
print(classification_report(y_test, y_pred, target_names=['No Diabetes','Diabetes']))

## STEP 10 – Confusion Matrix Plot

In [ ]:
# Visualize confusion matrix with custom labels
ConfusionMatrixDisplay(
    cm, display_labels=['No Diabetes', 'Diabetes']
).plot(cmap='Blues')
plt.title(f'KNN (k={best_k}) – Confusion Matrix')
plt.show()

---
# 📚 Practical Examination Explanation

## 🔧 Libraries Used

| Library | Purpose |
|---------|----------|
| **pandas** | Data manipulation, CSV loading, DataFrame operations |
| **numpy** | Numerical computations, array operations, mean calculations |
| **matplotlib.pyplot** | Basic plotting and visualization |
| **seaborn** | Statistical visualizations (heatmaps) |
| **sklearn.model_selection** | Train-test split functionality |
| **sklearn.preprocessing** | Feature scaling with StandardScaler |
| **sklearn.neighbors** | K-Nearest Neighbors classifier |
| **sklearn.metrics** | Evaluation metrics (accuracy, precision, recall, confusion matrix) |

---

## 📖 Technical Terms Explained

### 1. **K-Nearest Neighbors (KNN)**
- **Instance-based learning algorithm** that classifies based on the K closest training examples
- **Lazy learning**: Doesn't build a model during training, stores all training data
- **Distance-based**: Uses distance metrics (usually Euclidean) to find nearest neighbors
- **Voting mechanism**: Classification based on majority vote of K nearest neighbors

### 2. **Feature Scaling / Standardization**
```python
StandardScaler: X_scaled = (X - μ) / σ
```
- Transforms features to have **mean = 0** and **standard deviation = 1**
- **Critical for KNN** because distance calculations are sensitive to feature scales
- Example: Without scaling, BMI (20-40) would dominate over Insulin (0-1)

### 3. **Train-Test Split**
- **Purpose**: Evaluate model on unseen data to assess generalization
- **Common ratio**: 80% training, 20% testing
- **random_state**: Seed for reproducibility

### 4. **Confusion Matrix**
```
                 Predicted
              No(0)    Yes(1)
Actual No(0)   TN       FP
Actual Yes(1)  FN       TP
```
- **TN (True Negative)**: Correctly predicted no diabetes
- **TP (True Positive)**: Correctly predicted diabetes
- **FP (False Positive)**: Predicted diabetes when there was none (Type I error)
- **FN (False Negative)**: Missed diabetes diagnosis (Type II error)

### 5. **Evaluation Metrics**

#### **Accuracy**
```python
Accuracy = (TP + TN) / (TP + TN + FP + FN)
```
- Overall correctness of predictions
- **Limitation**: Can be misleading with imbalanced datasets

#### **Precision** (Positive Predictive Value)
```python
Precision = TP / (TP + FP)
```
- Of all predicted diabetic cases, how many are actually diabetic?
- **High precision**: Few false alarms

#### **Recall** (Sensitivity / True Positive Rate)
```python
Recall = TP / (TP + FN)
```
- Of all actual diabetic cases, how many did we detect?
- **High recall**: Few missed diagnoses
- **Critical in medical diagnosis**: Missing a diabetic patient is serious

#### **Error Rate**
```python
Error Rate = 1 - Accuracy = (FP + FN) / Total
```
- Proportion of incorrect predictions

### 6. **Classification Report**
- Provides precision, recall, F1-score for each class
- **F1-score**: Harmonic mean of precision and recall
- **Support**: Number of actual occurrences of each class

### 7. **Hyperparameter (K)**
- **K = number of neighbors** to consider for voting
- **Small K (e.g., 1-3)**: High variance, sensitive to noise, complex boundaries
- **Large K (e.g., 20-50)**: High bias, smoother boundaries, may miss patterns
- **Optimal K**: Found through cross-validation or error rate plotting

---

## ⚙️ K-Nearest Neighbors Algorithm

### How KNN Works

**Training Phase:**
1. Store all training data points with their labels
2. No actual "training" occurs (lazy learning)

**Prediction Phase:**
1. Calculate distance from new point to all training points
2. Select K nearest neighbors based on distance
3. For classification: Majority vote among K neighbors
4. For regression: Average of K neighbor values

### Distance Metrics

**Euclidean Distance** (most common):
```python
d = √[(x₁-x₂)² + (y₁-y₂)² + ... + (zₙ-z₂)²]
```

**Other metrics:**
- **Manhattan**: Sum of absolute differences
- **Minkowski**: Generalization of Euclidean and Manhattan

### Choosing Optimal K

**Methods:**
1. **Error rate plot**: Test K from 1 to n, plot error rates
2. **Cross-validation**: Use k-fold CV to find best K
3. **Rule of thumb**: K = √n (where n = number of training samples)

**Best practices:**
- Use **odd K** for binary classification to avoid ties
- **K should not be multiple of number of classes**
- Start with K = 5 and tune from there

---

## 🔄 Workflow Summary

```
1. Load Data → diabetes.csv
   ↓
2. Exploratory Data Analysis
   - Check shape, info, statistics
   - Visualize class distribution
   - Check for imbalance
   ↓
3. Data Cleaning
   - Replace zero values with mean (medical features can't be 0)
   - Verify no missing values
   ↓
4. Correlation Analysis
   - Heatmap to understand feature relationships
   - Identify strong predictors
   ↓
5. Feature-Target Split
   - X = All features except Outcome
   - y = Outcome (target)
   ↓
6. Train-Test Split (80-20)
   - Ensure reproducibility with random_state
   ↓
7. Feature Scaling (CRITICAL)
   - StandardScaler: mean=0, std=1
   - Fit on training, transform on both
   ↓
8. Hyperparameter Tuning
   - Test K values from 1 to 25
   - Plot error rates
   - Select K with minimum error
   ↓
9. Train Final Model
   - KNN with optimal K
   - Fit on training data
   ↓
10. Evaluation
    - Confusion matrix
    - Accuracy, Precision, Recall
    - Classification report
    - Visualize results
```

---

## 🏥 Real-World Applications

### Medical Diagnosis
- **Diabetes prediction** (this project)
- Heart disease classification
- Cancer detection
- Disease outbreak prediction

### Recommendation Systems
- Find K similar users/items
- Netflix movie recommendations
- Amazon product suggestions

### Pattern Recognition
- Handwritten digit recognition
- Face recognition
- Speech recognition

### Other Applications
- Credit risk assessment
- Customer segmentation
- Fraud detection
- Stock market prediction

---

## 🎯 Advantages & Limitations

### ✅ Advantages
1. **Simple to understand and implement**
2. **No training phase** (lazy learning)
3. **No assumptions** about data distribution
4. **Naturally handles multi-class** classification
5. **Effective with small datasets**

### ❌ Limitations
1. **Computationally expensive** for large datasets (calculates distances to all points)
2. **Memory intensive** (stores all training data)
3. **Sensitive to feature scaling** (requires standardization)
4. **Curse of dimensionality** (performance degrades with many features)
5. **Sensitive to irrelevant features** (no automatic feature selection)
6. **Imbalanced data issues** (majority class dominates voting)

---

## 🚀 Improvement Suggestions

### 1. **Handle Class Imbalance**
```python
from imblearn.over_sampling import SMOTE
smote = SMOTE(random_state=42)
X_train_balanced, y_train_balanced = smote.fit_resample(X_train, y_train)
```

### 2. **Weighted KNN**
```python
# Closer neighbors have more influence
knn = KNeighborsClassifier(n_neighbors=5, weights='distance')
```

### 3. **Different Distance Metrics**
```python
knn = KNeighborsClassifier(n_neighbors=5, metric='manhattan')
# Or: 'euclidean', 'minkowski', 'chebyshev'
```

### 4. **Cross-Validation for K Selection**
```python
from sklearn.model_selection import cross_val_score
for k in range(1, 26):
    knn = KNeighborsClassifier(n_neighbors=k)
    scores = cross_val_score(knn, X_train, y_train, cv=5)
    print(f'K={k}, Mean CV Score: {scores.mean():.4f}')
```

### 5. **Feature Selection**
```python
from sklearn.feature_selection import SelectKBest, f_classif
selector = SelectKBest(f_classif, k=5)
X_selected = selector.fit_transform(X_train, y_train)
```

### 6. **Dimensionality Reduction**
```python
from sklearn.decomposition import PCA
pca = PCA(n_components=5)
X_train_pca = pca.fit_transform(X_train)
```

### 7. **ROC Curve and AUC**
```python
from sklearn.metrics import roc_curve, roc_auc_score
y_proba = knn.predict_proba(X_test)[:, 1]
fpr, tpr, thresholds = roc_curve(y_test, y_proba)
auc = roc_auc_score(y_test, y_proba)
```

---

## 📝 Exam Preparation Tips

### Key Concepts to Remember

1. **Why scaling is mandatory for KNN?**
   - KNN uses distance calculations
   - Features with larger scales dominate the distance metric
   - Example: Income (50000) vs Age (35) - Income would dominate

2. **How to choose K?**
   - Plot error rate vs K value
   - Use cross-validation
   - Odd K for binary classification
   - Balance between bias and variance

3. **Medical diagnosis metrics priority**
   - **Recall** is more important than precision
   - Missing a diabetic patient (FN) is worse than false alarm (FP)
   - Better safe than sorry in medical context

4. **Confusion matrix interpretation**
   ```
   TN: Healthy correctly identified
   TP: Diabetic correctly identified
   FP: False alarm (healthy labeled diabetic)
   FN: Missed diagnosis (diabetic labeled healthy) ← Most serious
   ```

5. **KNN vs other algorithms**
   - **vs Logistic Regression**: KNN doesn't assume linear boundaries
   - **vs Decision Trees**: KNN is instance-based, trees are rule-based
   - **vs SVM**: KNN simpler but slower for large datasets
   - **vs Neural Networks**: KNN requires no training, NNs need training

### Common Exam Questions

**Q1: Why use StandardScaler instead of MinMaxScaler?**
- StandardScaler: Better for normally distributed data, handles outliers better
- MinMaxScaler: Scales to [0,1], sensitive to outliers

**Q2: What happens if K = n (total training samples)?**
- Model always predicts the majority class
- Becomes a constant classifier
- No learning occurs

**Q3: Can KNN handle multi-class classification?**
- Yes, naturally handles multiple classes
- Uses majority voting among K neighbors

**Q4: Why replace zeros with mean instead of median?**
- Both are valid imputation strategies
- Mean: Preserves overall distribution
- Median: More robust to outliers
- Choice depends on data distribution

**Q5: How to handle ties in KNN voting?**
- Use odd K for binary classification
- Use weighted voting (closer neighbors have more influence)
- Reduce K by 1 and re-vote

### Code Snippets to Remember

```python
# Essential imports
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score

# Basic KNN workflow
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train_scaled, y_train)
y_pred = knn.predict(X_test_scaled)

# Evaluation
cm = confusion_matrix(y_test, y_pred)
accuracy = accuracy_score(y_test, y_pred)
```

---

## 🎓 Summary

This practical demonstrates:
- ✅ **Binary classification** using K-Nearest Neighbors
- ✅ **Medical diagnosis** application (diabetes prediction)
- ✅ **Feature scaling** importance for distance-based algorithms
- ✅ **Hyperparameter tuning** to find optimal K
- ✅ **Comprehensive evaluation** with multiple metrics
- ✅ **Data preprocessing** (handling zeros/missing values)
- ✅ **Visualization** of class distribution, errors, and results

**Key Takeaway:** KNN is a simple yet powerful algorithm for classification tasks, especially effective with small-to-medium datasets when proper feature scaling is applied. For medical diagnosis, prioritize **recall** to minimize missed diagnoses.

---